# S3 Data Trigger for the Airbnb Price Pipeline — Event-Driven Retraining

**Module:** ITI113 Machine Learning & Operations
**Focus Area:** C — MLOps (Event-driven automation, monitoring, troubleshooting)
**Project:** Airbnb *fair advertised nightly rate* — **regression** on the continuous nightly `price`. Every trigger, pipeline step, metric and gate in this notebook serves that one continuous target.
**Estimated Runtime:** ~15–25 minutes per triggered SageMaker Pipeline execution on Studio

---

## What this notebook does

This notebook closes the MLOps loop that Notebooks 01–03 opened: **new data arriving should retrain the model without anyone editing a notebook.**

```text
New monthly CSV lands in the controlled folder
        s3://…/iti113/team14/trigger/input/
                      │
        ┌─────────────┴──────────────────┐
        │ DEV PATH (this notebook,       │ PRODUCTION PATH (Step 10)
        │ executed)                      │
        │ Jupyter polling loop detects   │ S3 → EventBridge rule
        │ the new object                 │   → Lambda handler
        └─────────────┬──────────────────┘   → StartPipelineExecution
                      ▼
        SageMaker Pipeline  (the SAME preprocess.py / train.py from Notebook 03,
                             uploaded CSV passed in as the InputDataUrl parameter)
                      ▼
        R²(log) quality gate ≥ 0.65
                      ▼
        Challenger vs champion comparison → Model Registry (new version)
                      ▼
        Promotion decision → @champion alias → CI/CD deployment automation (Step 11)
```

1. Establishes the **controlled trigger folder contract** — a dedicated prefix, an idempotency ledger, and a quarantine area.
2. Runs a **manual pipeline test first** (the reference notebook's discipline): validate the pipeline with an explicit parameter override before wiring any automation to it.
3. Executes the **simplified notebook monitoring loop** — the development-grade trigger: poll the folder, detect a genuinely new upload, start the pipeline, watch the steps.
4. On success, runs the **champion–challenger comparison** and registers the retrained model in the MLflow Model Registry with lineage, an approval status, and a governed promotion decision.
5. Stages a **deliberately broken upload** and diagnoses the failure programmatically — the local equivalent of CloudWatch/SageMaker log forensics, with the real `boto3` queries alongside.
6. Details the **production event-driven architecture** (S3 → EventBridge → Lambda → Pipeline) with the actual Lambda handler code and one-time infrastructure setup.
7. Explains how this trigger composes with the **CI/CD, registry and dashboard ecosystem** built in Notebook 03.

## Separate pipeline, same code

Like the reference workflow, the triggered pipeline gets its **own name** — `iti113-team14-airbnb-price-triggered` — keeping its executions cleanly separated from Notebook 03's manually-started `iti113-team14-airbnb-price`. What it runs, however, is *identical*: the same `pipeline_lib.py`, the same `train.py`. Event-driven retraining that runs different code from the validated pipeline would be a second train-serve-skew problem wearing an automation costume.

## AWS setup assumed on Studio

The team execution role can run Processing/Training Jobs, Pipelines, and Model Registry operations, with S3 access scoped to `iti113/team14/…`. The bucket has **EventBridge notifications enabled**; the rule and Lambda of Step 10 exist (or are created there once). CloudWatch Logs are available for Lambda and SageMaker jobs. Off Studio, everything below runs in **LOCAL_MODE**: a local folder mirrors the trigger prefix, the pipeline steps run as subprocesses of the same scripts, and per-step log files stand in for CloudWatch 

> **Prerequisites:** Notebook 03 completed in this workspace — `src/` (pipeline_lib, train.py), `artifacts/` (champion pipeline + evaluation report), `best_model.json`, and the MLflow registry holding `iti113-team14-airbnb-price-regressor` **v1 @champion**.

## Install or update required packages *(SageMaker Studio only)*

Uncomment on Studio and restart the kernel after installing.

In [1]:
# %pip install --upgrade "sagemaker>=2,<3" boto3 botocore
# %pip install --upgrade mlflow sagemaker-mlflow

## 0. Configuration

Only two genuinely new settings exist here — the **trigger folder** and the **promotion tolerance** — everything else is inherited from Notebooks 01–03 so the trigger operates on the *same* project, bucket layout, gate and registry. Note the reference notebook's separation of prefixes, preserved exactly: uploads to `manual-input/` never wake the automation; only `trigger/input/` is watched.

In [2]:
import os, io, sys, json, glob, time, shutil, hashlib, tempfile, subprocess, warnings, re, contextlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# Student / team settings — change only these
# ============================================================
TEAM_ID      = "team14"
STUDENT_ID   = "s1402"
COURSE       = "ITI113"
SEMESTER     = "26S1"
PROJECT_NAME = "airbnb-listings"
REGION       = "ap-southeast-1"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

RANDOM_STATE = 42
TEST_SIZE    = 0.20
QUALITY_GATE_R2LOG  = 0.65     # absolute floor, unchanged since Notebook 02
PROMOTION_TOLERANCE = 0.005    # challenger may trail champion R2_log by at most this (freshness credit)

# --- Event-driven naming (separate from the Notebook 03 manual pipeline)
PIPELINE_NAME           = f"iti113-{TEAM_ID}-airbnb-price"              # NB03 manual pipeline
TRIGGERED_PIPELINE_NAME = f"iti113-{TEAM_ID}-airbnb-price-triggered"    # this notebook
REGISTRY_MODEL_NAME     = f"iti113-{TEAM_ID}-airbnb-price-regressor"
MODEL_PACKAGE_GROUP     = f"iti113-{TEAM_ID}-airbnb-price-models"
LAMBDA_FUNCTION_NAME    = f"iti113-{TEAM_ID}-airbnb-trigger"
EVENTBRIDGE_RULE_NAME   = f"iti113-{TEAM_ID}-airbnb-csv-upload"

# --- The controlled folders (reference-notebook discipline: trigger vs manual kept apart)
TRIGGER_PREFIX     = f"iti113/{TEAM_ID}/trigger/input"       # watched — uploads HERE retrain
MANUAL_TEST_PREFIX = f"iti113/{TEAM_ID}/manual-input"        # not watched — safe testing ground
QUARANTINE_PREFIX  = f"iti113/{TEAM_ID}/trigger/quarantine"  # failed uploads parked for forensics

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ============================================================
# Environment detection: SageMaker vs local
# ============================================================
try:
    import sagemaker, boto3
    session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    LOCAL_MODE = False
    print(f"SageMaker mode | role: {role.split('/')[-1]}")
except Exception:
    LOCAL_MODE = True
    print("SageMaker not available -> LOCAL_MODE (a local folder mirrors the trigger prefix; "
          "the SAME pipeline scripts run as subprocesses; per-step log files stand in for CloudWatch)")

# Local mirrors of the controlled folders + per-event run workspace
S3_MIRROR       = Path("s3_mirror")
WATCH_DIR       = S3_MIRROR / TRIGGER_PREFIX
MANUAL_DIR      = S3_MIRROR / MANUAL_TEST_PREFIX
QUARANTINE_DIR  = S3_MIRROR / QUARANTINE_PREFIX
RUNS_DIR        = Path("pipeline_runs")          # one folder per execution: logs + processed + artifacts
STATE_FILE      = Path("processed_uploads.json") # the idempotency ledger
LOCAL_ARTIFACTS = "artifacts"                    # Notebook 03's champion artifacts

# ============================================================
# Notebook 03 hand-off checks
# ============================================================
for req in ["src/pipeline_lib.py", "src/train.py", f"{LOCAL_ARTIFACTS}/model_pipeline.joblib",
            f"{LOCAL_ARTIFACTS}/evaluation_report.json", "best_model.json"]:
    if not os.path.exists(req):
        raise FileNotFoundError(f"{req} not found — run Notebook 03 in this workspace first.")
BEST = json.loads(Path("best_model.json").read_text())
CHAMPION_REPORT = json.loads(Path(LOCAL_ARTIFACTS, "evaluation_report.json").read_text())
CHAMPION_HPARAMS = CHAMPION_REPORT["hyperparameters"]

MLFLOW_EXPERIMENT = f"{COURSE}/{TEAM_ID}/Experiment1"
if LOCAL_MODE:
    MLFLOW_TRACKING_URI = "sqlite:///mlflow_local.db"
else:
    cands = ([Path(f"mlflow_app_config_{TEAM_ID}.json"),
              Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")]
             + [Path(p) for p in sorted(glob.glob(f"mlflow_app_config_{TEAM_ID}_*.json"))])
    arn = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS"
    if arn is None:
        raise RuntimeError("No MLflow App config found — run Notebook 01A or place the team config JSON here.")
    MLFLOW_TRACKING_URI = arn

print(f"Watched folder    : {f's3://{BUCKET}/{TRIGGER_PREFIX}/' if not LOCAL_MODE else WATCH_DIR}/")
print(f"Manual test folder: {f's3://{BUCKET}/{MANUAL_TEST_PREFIX}/' if not LOCAL_MODE else MANUAL_DIR}/")
print(f"Triggered pipeline: {TRIGGERED_PIPELINE_NAME}")
print(f"Registry          : {REGISTRY_MODEL_NAME}")
print(f"Gate / promotion  : R2_log >= {QUALITY_GATE_R2LOG}  |  challenger >= champion - {PROMOTION_TOLERANCE}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SageMaker mode | role: SageMakerExecutionRole-ITI113-Team14
Watched folder    : s3://nyp-26s1-iti113/iti113/team14/trigger/input//
Manual test folder: s3://nyp-26s1-iti113/iti113/team14/manual-input//
Triggered pipeline: iti113-team14-airbnb-price-triggered
Registry          : iti113-team14-airbnb-price-regressor
Gate / promotion  : R2_log >= 0.65  |  challenger >= champion - 0.005


## 0A. Connect to MLflow and load the reigning champion

The trigger's whole purpose is to challenge the incumbent, so the first act is to establish exactly *who* the incumbent is — resolved live from the **`@champion` registry alias**, never from a hard-coded run id. The same `TeamId` tag check as Notebooks 02–03 guards the SageMaker MLflow App before anything is written.

In [3]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

if not LOCAL_MODE:
    sm_client = boto3.client("sagemaker", region_name=REGION)
    tags = {t["Key"]: t["Value"] for t in sm_client.list_tags(ResourceArn=MLFLOW_TRACKING_URI).get("Tags", [])}
    if tags.get("TeamId") != TEAM_ID:
        raise PermissionError(f"MLflow App TeamId={tags.get('TeamId')} does not match {TEAM_ID}.")
    print(f"[OK] MLflow App tag TeamId={TEAM_ID}")
else:
    print("[SKIP] LOCAL_MODE — using the local SQLite store shared with Notebooks 02-03.")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = MlflowClient()

COMMON_TAGS = {
    "course": COURSE, "semester": SEMESTER, "team_id": TEAM_ID, "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME, "task_type": "regression",
    "tracking_backend": "local_sqlite" if LOCAL_MODE else "sagemaker_mlflow_app",
    "notebook": "s3-data-trigger-pipeline",
}

def load_champion():
    mv = client.get_model_version_by_alias(REGISTRY_MODEL_NAME, "champion")
    run = mlflow.get_run(mv.run_id)
    return {"version": int(mv.version), "run_id": mv.run_id,
            "approval": mv.tags.get("approval_status"),
            "data_version": mv.tags.get("data_version"),
            "metrics": {k: v for k, v in run.data.metrics.items() if k.startswith("test_")}}

CHAMPION = load_champion()
print(f"Reigning champion: {REGISTRY_MODEL_NAME} v{CHAMPION['version']} "
      f"(approval={CHAMPION['approval']}, data={CHAMPION['data_version']})")
for m in ["test_mae_log", "test_rmse_log", "test_r2_log", "test_mape"]:
    print(f"  {m:15s} {CHAMPION['metrics'][m]}")

[OK] MLflow App tag TeamId=team14


Reigning champion: iti113-team14-airbnb-price-regressor v7 (approval=approved, data=sha256:097b0bbfea3c)
  test_mae_log    0.3207
  test_rmse_log   0.451
  test_r2_log     0.8708
  test_mape       33.41


## 1. The Controlled Folder Contract

"Uploading a CSV retrains the model" is a sharp knife, so the folder it watches is governed by four rules:

| Rule | Mechanism | Why |
|---|---|---|
| **Only one prefix triggers** | `trigger/input/` is watched; `manual-input/` and everything else is inert | Accidental uploads elsewhere must never burn a training run — the reference notebook's manual/trigger separation |
| **Exactly-once processing** | `processed_uploads.json` — an idempotency ledger keyed by object key + size, checked before every start | Pollers re-list and EventBridge is *at-least-once*: duplicate events must not mean duplicate retrains |
| **Failures are quarantined, not retried** | failed uploads move to `trigger/quarantine/` with their logs | A schema-broken file would fail identically forever; retry loops on poison inputs are an outage generator |
| **Uploads are named snapshots** | `Listings_YYYY-MM.csv` | The filename becomes part of the run's lineage; the content hash (`data_version`) remains the ground truth |

The raw zone stays immutable: the trigger folder is an *inbox*, and the pipeline's processing step reads the uploaded object directly via the `InputDataUrl` parameter — nothing overwrites `raw/Listings.csv`.

In [4]:
for d in (WATCH_DIR, MANUAL_DIR, QUARANTINE_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}

def save_state(state):
    STATE_FILE.write_text(json.dumps(state, indent=2))

def upload_key(path):
    p = Path(path)
    return f"{p.name}|{p.stat().st_size}"

state = load_state()
print(f"Controlled folders ready under {S3_MIRROR}/ "
      f"(mirroring s3://{BUCKET}/iti113/{TEAM_ID}/...)")
for d in (WATCH_DIR, MANUAL_DIR, QUARANTINE_DIR):
    print(f"  {d}/")
print(f"Idempotency ledger: {STATE_FILE} ({len(state)} uploads processed so far)")
print(f"Per-execution workspace: {RUNS_DIR}/<event_id>/  (logs, processed splits, artifacts)")

Controlled folders ready under s3_mirror/ (mirroring s3://nyp-26s1-iti113/iti113/team14/...)
  s3_mirror/iti113/team14/trigger/input/
  s3_mirror/iti113/team14/manual-input/
  s3_mirror/iti113/team14/trigger/quarantine/
Idempotency ledger: processed_uploads.json (2 uploads processed so far)
Per-execution workspace: pipeline_runs/<event_id>/  (logs, processed splits, artifacts)


## 2. Pipeline Parameters — `InputDataUrl` Takes Center Stage

Building on the core design of our reference notebook, **the dataset is no longer hardcoded into the pipeline. It is injected dynamically as a runtime parameter.** Whenever a new dataset is uploaded, a Lambda function (or a local developer loop) triggers a new pipeline execution, passing the uploaded file's location directly into `InputDataUrl`. Because our champion hyperparameters and quality gate thresholds are passed the exact same way, we can trigger a complete retrain with new data or settings without ever modifying the underlying pipeline code.

This design ensures perfect local-to-cloud parity. When running locally, these pipeline parameters map one-to-one with the CLI arguments used by `preprocess.py` and `train.py`. As a result, a single parameter dictionary seamlessly drives execution across both environments.

In [5]:
PIPELINE_PARAMETERS = {
    "InputDataUrl":     None,   # set per event: the uploaded CSV (S3 URI or local path)
    "LearningRate":     CHAMPION_HPARAMS["learning_rate"],
    "MaxLeafNodes":     CHAMPION_HPARAMS["max_leaf_nodes"],
    "MaxIter":          CHAMPION_HPARAMS["max_iter"],
    "MinSamplesLeaf":   CHAMPION_HPARAMS["min_samples_leaf"],
    "L2Regularization": CHAMPION_HPARAMS["l2_regularization"],
    "QualityGateR2Log": QUALITY_GATE_R2LOG,
}
print("Pipeline parameters (defaults = reigning champion configuration):")
for k, v in PIPELINE_PARAMETERS.items():
    print(f"  {k:17s} = {v if v is not None else '<set per triggering upload>'}")

Pipeline parameters (defaults = reigning champion configuration):
  InputDataUrl      = <set per triggering upload>
  LearningRate      = 0.1
  MaxLeafNodes      = 127
  MaxIter           = 300
  MinSamplesLeaf    = 50
  L2Regularization  = 1.0
  QualityGateR2Log  = 0.65


## 3. One Backwards-Compatible Script Change

When triggered, the pipeline reuses the scripts from Notebook 03 entirely **unchanged**, with just one minor adjustment. In SageMaker, a `ProcessingInput` that points to an uploaded file actually mounts it inside a directory (`/opt/ml/processing/input/<filename>`). Because the exact filename isn't known when defining the pipeline, we implement a globbing solution to locate the data dynamically. 

To handle this, `preprocess.py` is updated to **v1.1**: its `--input` argument is now designed to accept either a direct file path *or* a directory path (in which case it automatically selects the newest `*.csv` file). Beyond this single input-handling tweak, everything downstream—the `pipeline_lib`, data cleaning logic, train/test splitting, `train.py`, and `inference.py`—remains byte-for-byte identical to the versions proven reliable in the Notebook 03 consistency checks.

In [6]:
%%writefile src/preprocess.py
"""SageMaker Processing Job — versioned cleaning + split for the Airbnb price regressor.

v1.1: --input accepts a file OR a directory (newest *.csv inside wins). The triggered
pipeline mounts the uploaded S3 object into a directory, so the entrypoint resolves it;
explicit file paths (Notebook 03 and local runs) behave exactly as before.

Reads the snapshot named by InputDataUrl, applies the Notebook 01 auditable cleaning
sequence (via pipeline_lib), performs the city-stratified split, and writes the
versioned cleaned datasets + a data manifest to /processed.
"""
import argparse
import glob as globlib
import json
import os
import sys
from datetime import datetime, timezone

HERE = os.path.dirname(os.path.abspath(__file__))
for p in (HERE, "/opt/ml/processing/input/code"):
    if p not in sys.path:
        sys.path.append(p)

import pandas as pd
from sklearn.model_selection import train_test_split
from pipeline_lib import clean_listings, get_data_version, __version__ as LIB_VERSION

parser = argparse.ArgumentParser()
parser.add_argument("--input", type=str, default="/opt/ml/processing/input")
parser.add_argument("--output-dir", type=str, default="/opt/ml/processing/output")
parser.add_argument("--test-size", type=float, default=0.20)
parser.add_argument("--random-state", type=int, default=42)
args = parser.parse_args()
os.makedirs(args.output_dir, exist_ok=True)

# v1.1 input resolution: file path, or directory containing the uploaded CSV
input_path = args.input
if os.path.isdir(input_path):
    csvs = sorted(globlib.glob(os.path.join(input_path, "**", "*.csv"), recursive=True),
                  key=os.path.getmtime)
    if not csvs:
        raise FileNotFoundError(f"No CSV found under {input_path}")
    input_path = csvs[-1]

import numpy as np
import sklearn
print(f"pipeline_lib v{LIB_VERSION} | pandas {pd.__version__} | numpy {np.__version__} "
      f"| sklearn {sklearn.__version__}", flush=True)
print(f"resolved input: {input_path}", flush=True)

# The sklearn 1.2-1 container ships pandas 1.1.x, which predates the
# encoding_errors kwarg (pandas >= 1.3). Handing pandas a file object opened
# with errors="replace" is behaviourally identical on the older line.
try:
    df_raw = pd.read_csv(input_path, encoding="utf-8", encoding_errors="replace",
                         low_memory=False)
except TypeError:
    with open(input_path, "r", encoding="utf-8", errors="replace", newline="") as _fh:
        df_raw = pd.read_csv(_fh, low_memory=False)
data_version = get_data_version(input_path)
print(f"raw rows: {len(df_raw):,} | data_version: {data_version}")

df, audit = clean_listings(df_raw)
for a in audit:
    print(f"  clean: {a['step']:28s} -{a['rows_removed']:>4} rows  ({a['note']})")
print(f"clean rows: {len(df):,} ({len(df_raw) - len(df)} removed)")

train_df, test_df = train_test_split(df, test_size=args.test_size,
                                     random_state=args.random_state, stratify=df["city"])
train_df.to_csv(os.path.join(args.output_dir, "train.csv"), index=False)
test_df.to_csv(os.path.join(args.output_dir, "test.csv"), index=False)

manifest = {
    "data_version": data_version,
    "source_file": os.path.basename(input_path),
    "raw_rows": int(len(df_raw)), "clean_rows": int(len(df)),
    "train_rows": int(len(train_df)), "test_rows": int(len(test_df)),
    "test_size": args.test_size, "random_state": args.random_state,
    "stratify": "city", "audit": audit, "pipeline_lib_version": LIB_VERSION,
    "columns": df.columns.tolist(),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
with open(os.path.join(args.output_dir, "data_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"train: {len(train_df):,} rows | test: {len(test_df):,} rows")
print("wrote train.csv, test.csv, data_manifest.json")


Overwriting src/preprocess.py


In [7]:
print("Pipeline source of truth (only preprocess.py changed, v1.0 -> v1.1):")
for fn in ["pipeline_lib.py", "preprocess.py", "train.py", "inference.py"]:
    print(f"  src/{fn:16s} {os.path.getsize(f'src/{fn}'):>7,} bytes"
          + ("   <- v1.1: directory-aware --input" if fn == "preprocess.py" else ""))
sys.path.insert(0, os.path.abspath("src"))
from pipeline_lib import get_data_version, SERVING_COLUMNS, __version__ as LIB_VERSION
print(f"pipeline_lib v{LIB_VERSION} imported — transformation logic untouched since the "
      f"Notebook 03 train-serve consistency proof.")

Pipeline source of truth (only preprocess.py changed, v1.0 -> v1.1):
  src/pipeline_lib.py   18,013 bytes
  src/preprocess.py      3,814 bytes   <- v1.1: directory-aware --input
  src/train.py           5,208 bytes
  src/inference.py       1,665 bytes


pipeline_lib v1.0.0 imported — transformation logic untouched since the Notebook 03 train-serve consistency proof.


## 4. The Retraining Runner — Single Execution, Full Observability

The `run_retraining_pipeline()` function acts as the local counterpart to SageMaker's `StartPipelineExecution`. It executes the **exact same two steps** (`PreprocessData` → `TrainModel`) as the managed cloud pipeline. Each step is spun up as a subprocess using our unmodified scripts, isolated within a private, per-event workspace (`pipeline_runs/<event_id>/`). 

To perfectly mirror SageMaker's observability, the runner enforces three guarantees:

*   **Dedicated Log Files:** Every step's stdout and stderr streams are routed directly to their **own log file**, acting as a local simulation of a CloudWatch log stream.
*   **Execution Tracking:** A comprehensive **step-status table** records the `StepName`, `StepStatus`, `duration`, and `FailureReason`—providing a local stand-in for the `list_pipeline_execution_steps` API.
*   **Automated Failure Extraction:** If a step fails, the runner automatically **extracts the core failure reason from the log tail**, replicating precisely how SageMaker populates its `FailureReason` surface.

Crucially, this runner does not re-implement or interfere with any underlying pipeline logic. It exists solely to orchestrate and observe the process, which is exactly the boundary a robust trigger should maintain

In [8]:
def _tail_error(log_path, n=30):
    """Extract the most informative failure line from a step log (what SageMaker
    surfaces as FailureReason)."""
    lines = Path(log_path).read_text(errors="replace").splitlines()
    for line in reversed(lines[-n:]):
        if re.search(r"\b\w*(Error|Exception)\b", line):
            return line.strip()
    return lines[-1].strip() if lines else "no output"

def run_retraining_pipeline(csv_path, event_id, params=None):
    """Local twin of StartPipelineExecution: PreprocessData -> TrainModel on the
    uploaded CSV, per-event workspace, per-step logs, step-status records."""
    params = dict(PIPELINE_PARAMETERS, **(params or {}), InputDataUrl=str(csv_path))
    run_dir = RUNS_DIR / event_id
    processed, artifacts, logs = run_dir / "processed", run_dir / "artifacts", run_dir / "logs"
    for d in (processed, artifacts, logs):
        d.mkdir(parents=True, exist_ok=True)

    print(f"Execution: {TRIGGERED_PIPELINE_NAME}/{event_id}")
    print(f"PipelineParameters: InputDataUrl={Path(str(csv_path)).name}, "
          f"MaxIter={params['MaxIter']}, QualityGateR2Log={params['QualityGateR2Log']}")

    step_cmds = [
        ("PreprocessData", [sys.executable, "src/preprocess.py",
                            "--input", str(csv_path), "--output-dir", str(processed),
                            "--test-size", str(TEST_SIZE), "--random-state", str(RANDOM_STATE)]),
        ("TrainModel",     [sys.executable, "src/train.py",
                            "--learning-rate", str(params["LearningRate"]),
                            "--max-leaf-nodes", str(params["MaxLeafNodes"]),
                            "--max-iter", str(params["MaxIter"]),
                            "--min-samples-leaf", str(params["MinSamplesLeaf"]),
                            "--l2-regularization", str(params["L2Regularization"]),
                            "--random-state", str(RANDOM_STATE),
                            "--gate-r2", str(params["QualityGateR2Log"]),
                            "--team-id", TEAM_ID, "--student-id", STUDENT_ID,
                            "--train", str(processed), "--test", str(processed),
                            "--model-dir", str(artifacts)]),
    ]
    steps = []
    for name, cmd in step_cmds:
        log_path = logs / f"{name}.log"
        print(f"  {name:<16} Executing ...", end="", flush=True)
        t0 = time.time()
        with open(log_path, "w") as lf:
            rc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT).returncode
        status = "Succeeded" if rc == 0 else "Failed"
        rec = {"StepName": name, "StepStatus": status, "seconds": round(time.time() - t0, 1),
               "log": str(log_path),
               "FailureReason": "" if rc == 0 else _tail_error(log_path)}
        steps.append(rec)
        print(f"\r  {name:<16} {status}  ({rec['seconds']}s)"
              + (f"\n                   FailureReason: {rec['FailureReason']}" if rc else ""))
        if rc != 0:
            break
    ok = all(s["StepStatus"] == "Succeeded" for s in steps) and len(steps) == len(step_cmds)
    return {"event_id": event_id, "input": str(csv_path), "run_dir": str(run_dir),
            "status": "Succeeded" if ok else "Failed", "steps": steps,
            "parameters": params}

def print_step_table(execution):
    print(f"Pipeline status: {execution['status']}   ({execution['event_id']})")
    for s in execution["steps"]:
        line = f"  - {s['StepName']:<16} {s['StepStatus']:<10} {s['seconds']:>6.1f}s"
        if s["FailureReason"]:
            line += f"\n      FailureReason: {s['FailureReason']}"
        print(line)

print("Runner ready: run_retraining_pipeline(csv, event_id) -> step-status records "
      "+ per-step logs under pipeline_runs/<event_id>/logs/")

Runner ready: run_retraining_pipeline(csv, event_id) -> step-status records + per-step logs under pipeline_runs/<event_id>/logs/


### 4.1 Beyond the Gate: Challenger vs. Champion and the Model Registry

Successfully passing the absolute quality gate is only the first hurdle; a new model (the "challenger") must still justify its place against the current production model (the "champion"). The post-run handler automates this evaluation and executes the outcome directly within our governance layer—the Model Registry:

1.  **Log the Retrain to MLflow:** The new run is logged with the tag `run_type=event_triggered_retrain` and includes the triggering key. We capture the content-addressed `data_version` (the hash of the new snapshot), all hyperparameters, performance metrics, and a visual challenger-vs-champion comparison.
2.  **Register the Challenger (Always):** Every newly trained model is registered as a new version, even if it ultimately loses. This guarantees a complete, unbroken audit history. The model is initially tagged with `approval_status=pending_review`, alongside its lineage and gate metrics.
3.  **Policy-Driven Promotion:** Promotion decisions are dictated by strict mathematical policy, not subjective judgment. A challenger is promoted *if and only if* it meets the absolute floor (`R²_log ≥ 0.65`) **AND** achieves non-inferiority (`challenger ≥ champion − 0.005`). We allow this tiny margin of error because *fresher data carries intrinsic value*. Staleness is a silent performance killer (as documented in Notebook 03 regarding Istanbul's TRY inflation), so a statistically flat model trained on newer data should win. If the challenger meets these criteria, its status flips to `approved` and the **`@champion`** alias is reassigned. If it fails, it remains stuck in `pending_review` for human investigation.

**A Note on Evaluation Fairness:** 
It is important to transparently state that each model is evaluated on the holdout set generated from **its own data snapshot** (using the exact same splitting recipe on contemporaneous data). This represents a standard rolling-refresh comparison. The alternative is a permanently frozen reference test set—would provide perfect one-to-one comparability, but at the steep cost of rapidly aging out of real-world relevance. We accept the rolling-refresh trade-off here. 

In [9]:
EVENT_LOG = []

def evaluate_and_register(execution, auto_promote=True):
    """Champion-challenger comparison + registry write for a Succeeded execution."""
    global CHAMPION
    run_dir = Path(execution["run_dir"])
    report = json.loads((run_dir / "artifacts" / "evaluation_report.json").read_text())
    manifest = json.loads((run_dir / "processed" / "data_manifest.json").read_text())
    ch, inc = report["metrics"], CHAMPION["metrics"]

    # --- the comparison table
    comp = pd.DataFrame({
        f"champion v{CHAMPION['version']}": [inc[m] for m in
            ["test_mae_log", "test_rmse_log", "test_r2_log", "test_mape"]],
        "challenger": [ch[m] for m in ["test_mae_log", "test_rmse_log", "test_r2_log", "test_mape"]],
    }, index=["test_mae_log", "test_rmse_log", "test_r2_log", "test_mape"])
    comp["delta"] = (comp["challenger"] - comp.iloc[:, 0]).round(4)
    print("\nChampion vs challenger (each on its own snapshot's holdout):")
    print(comp.to_string())

    gate_ok = report["quality_gate"]["passed"]
    non_inferior = ch["test_r2_log"] >= inc["test_r2_log"] - PROMOTION_TOLERANCE
    promote = bool(auto_promote and gate_ok and non_inferior)
    decision = ("PROMOTE (gate passed, within non-inferiority tolerance of champion)"
                if promote else
                "HOLD as pending_review (" +
                ("quality gate failed" if not gate_ok else
                 f"trails champion R2_log by > {PROMOTION_TOLERANCE}") + ")")

    # --- per-city figure: champion vs challenger MAPE
    pc_ch = pd.DataFrame(report["per_city"]).set_index("city")["mape_pct"]
    pc_in = pd.DataFrame(CHAMPION_REPORT["per_city"]).set_index("city")["mape_pct"]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    x = np.arange(len(pc_in))
    ax.bar(x - 0.2, pc_in.values, 0.4, label=f"champion v{CHAMPION['version']}", color="#5B9BD5")
    ax.bar(x + 0.2, pc_ch.reindex(pc_in.index).values, 0.4, label="challenger", color="#ED7D31")
    ax.set_xticks(x); ax.set_xticklabels(pc_in.index, rotation=30, ha="right")
    ax.set_ylabel("MAPE %"); ax.legend(frameon=False)
    ax.set_title("Per-city MAPE — event-triggered challenger vs reigning champion", fontweight="bold")
    plt.tight_layout()

    # --- MLflow run for the retrain
    with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_retrain_{execution['event_id']}") as run:
        mlflow.set_tags({**COMMON_TAGS, "run_type": "event_triggered_retrain",
                         "trigger_key": Path(execution["input"]).name,
                         "event_id": execution["event_id"],
                         "data_version": manifest["data_version"]})
        mlflow.log_params({f"hp_{k}": v for k, v in report["hyperparameters"].items()})
        mlflow.log_params({"input_data_url": execution["input"],
                           "data_version": manifest["data_version"],
                           "n_train": report["n_train"], "n_test": report["n_test"],
                           "champion_version_at_trigger": CHAMPION["version"],
                           "pipeline_lib_version": report["pipeline_lib_version"]})
        mlflow.log_metrics({**{k: v for k, v in report["metrics"].items()},
                            "fit_seconds": report["fit_seconds"],
                            "champion_test_r2_log": inc["test_r2_log"],
                            "r2_log_delta_vs_champion": round(ch["test_r2_log"] - inc["test_r2_log"], 4)})
        with tempfile.TemporaryDirectory() as tmp:
            fig.savefig(f"{tmp}/champion_vs_challenger.png", dpi=110, bbox_inches="tight")
            comp.to_csv(f"{tmp}/comparison.csv")
            mlflow.log_artifact(f"{tmp}/champion_vs_challenger.png", artifact_path="comparison")
            mlflow.log_artifact(f"{tmp}/comparison.csv", artifact_path="comparison")
            mlflow.log_artifact(str(run_dir / "artifacts" / "evaluation_report.json"),
                                artifact_path="comparison")
        import joblib
        mlflow.sklearn.log_model(
            joblib.load(run_dir / "artifacts" / "model_pipeline.joblib"),
            name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            code_paths=["src/pipeline_lib.py"])
        retrain_run_id = run.info.run_id
    plt.show(); plt.close()

    # --- register the challenger (win or lose: versions are audit history)
    with contextlib.redirect_stderr(io.StringIO()):
        mv = mlflow.register_model(f"runs:/{retrain_run_id}/model", REGISTRY_MODEL_NAME)
    for k, v in {"approval_status": "pending_review", "trigger_key": Path(execution["input"]).name,
                 "event_id": execution["event_id"], "data_version": manifest["data_version"],
                 "gate_metric": "test_r2_log", "gate_threshold": str(QUALITY_GATE_R2LOG),
                 "gate_value": str(ch["test_r2_log"]),
                 "challenged_champion": f"v{CHAMPION['version']}"}.items():
        client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, k, v)
    print(f"\nRegistered challenger: {REGISTRY_MODEL_NAME} v{mv.version} (pending_review)")
    print(f"Promotion policy     : gate>={QUALITY_GATE_R2LOG} [{'PASS' if gate_ok else 'FAIL'}] "
          f"AND R2_log >= champion-{PROMOTION_TOLERANCE} [{'PASS' if non_inferior else 'FAIL'}]")
    print(f"Decision             : {decision}")

    if promote:
        client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "approval_status", "approved")
        client.set_model_version_tag(REGISTRY_MODEL_NAME, mv.version, "approved_by",
                                     f"promotion-policy/{TEAM_ID}")
        client.set_model_version_tag(REGISTRY_MODEL_NAME, CHAMPION["version"],
                                     "superseded_by", f"v{mv.version}")
        client.set_registered_model_alias(REGISTRY_MODEL_NAME, "champion", mv.version)
        print(f"@champion alias      : v{CHAMPION['version']} -> v{mv.version} "
              f"(deployment automation reacts to this — Notebook 03 Step 10)")
        CHAMPION = load_champion()

    return {"retrain_run_id": retrain_run_id, "model_version": int(mv.version),
            "promoted": promote, "decision": decision, "metrics": ch}

print("Post-run handler ready: evaluate_and_register(execution) -> "
      "MLflow run + registry version + governed promotion decision")

Post-run handler ready: evaluate_and_register(execution) -> MLflow run + registry version + governed promotion decision


## 5. Create the New Data Snapshot

Our setup simulates the real-world event this system was built to handle: a **fresh monthly platform extract**. 

We use `Listings_2021-04.csv`, which represents a March data snapshot featuring roughly 10% listing churn (delisted properties are removed, while everything else remains untouched). This produces a file that is completely schema-identical but content-different—exactly what you would expect from a standard monthly data dump. Because its SHA-256 hash differs from the champion model's original training data, the retrain's `data_version` will automatically track and separate the lineage of the two snapshots.

Crucially, this new file is initially generated in a **staging area** rather than the actively watched folder. This deliberate safeguard allows us to manually test the pipeline in Step 6 before giving any automation the green light to execute.

In [10]:
STAGING = S3_MIRROR / "staging"
STAGING.mkdir(parents=True, exist_ok=True)
SNAPSHOT_APRIL = STAGING / "Listings_2021-04.csv"

raw = pd.read_csv("Listings.csv" if os.path.exists("Listings.csv")
                  else "/mnt/user-data/uploads/Listings.csv",
                  encoding="utf-8", encoding_errors="replace", low_memory=False)
rng = np.random.RandomState(1404)                      # April churn seed
keep = rng.rand(len(raw)) >= 0.10                      # ~10% of listings churn out
raw[keep].to_csv(SNAPSHOT_APRIL, index=False)

snap = pd.read_csv(SNAPSHOT_APRIL, nrows=5, low_memory=False)
assert list(snap.columns) == list(raw.columns), "snapshot schema must match the raw extract"
print(f"Created {SNAPSHOT_APRIL.name}: {int(keep.sum()):,} rows "
      f"({(~keep).sum():,} listings churned out of {len(raw):,}) | "
      f"{SNAPSHOT_APRIL.stat().st_size/1e6:.0f} MB")
print(f"Schema: {len(snap.columns)} columns, identical to the March extract")
print(f"data_version: {get_data_version(SNAPSHOT_APRIL)}  "
      f"(champion trained on {CHAMPION['data_version']})")
del raw

Created Listings_2021-04.csv: 251,625 rows (28,087 listings churned out of 279,712) | 146 MB
Schema: 33 columns, identical to the March extract


data_version: sha256:c0d173e2380a  (champion trained on sha256:097b0bbfea3c)


## 5A. Create the Triggered Pipeline

`Step 6` and `Step 7` start `TRIGGERED_PIPELINE_NAME`, but nothing has created it yet, hence `ResourceNotFound`. This cell publishes the v1.1 scripts and upserts a pipeline whose `InputDataUrl` parameter names the snapshot to retrain on.

In [11]:
# ================================================================
# 5A. Create the triggered pipeline  (MUST run before Step 6 / Step 7)
# ----------------------------------------------------------------
# Nothing in this notebook has ever created iti113-<team>-airbnb-price-triggered.
# Step 6 and Step 7 call start_pipeline_execution against it, which is why the manual
# test died with ResourceNotFound.
#
# It is deliberately a SECOND pipeline, not a reuse of Notebook 03's:
#   - NB03's pipeline is the graded manual artifact, pinned to the immutable
#     /raw extract. Its execution history should stay clean.
#   - This one takes InputDataUrl and retrains on arbitrary uploaded snapshots.
# The step graph is otherwise identical, so the comparison stays honest.
# ================================================================
if LOCAL_MODE:
    print("[SKIP] LOCAL_MODE — the managed pipeline is created on Studio. "
          "Locally, run_retraining_pipeline() simulates this same graph with subprocesses.")
else:
    import sagemaker
    from sagemaker.workflow.pipeline import Pipeline
    from sagemaker.workflow.steps import ProcessingStep, TrainingStep
    from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
    from sagemaker.workflow.condition_step import ConditionStep
    from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
    from sagemaker.workflow.model_step import ModelStep
    from sagemaker.workflow.pipeline_context import PipelineSession
    from sagemaker.processing import FrameworkProcessor, ProcessingInput, ProcessingOutput
    from sagemaker.sklearn.estimator import SKLearn
    from sagemaker.model import Model
    from pathlib import Path

    sm_client = boto3.client("sagemaker", region_name=REGION)
    pipeline_session = PipelineSession()
    PIPELINE_ROOT      = f"s3://{BUCKET}/{PREFIX}/processed/triggered"
    SCRIPTS_S3_URI     = f"s3://{BUCKET}/{PREFIX}/pipeline_src_v11"
    LOCAL_PIPELINE_SRC = "pipeline_src_v11"
    DEFAULT_INPUT_URI  = f"s3://{BUCKET}/{PREFIX}/raw/Listings.csv"

    # --- Publish the v1.1 scripts, then pull them into a clean folder.
    # Round-tripping through S3 (rather than packaging src/ directly) is what
    # makes "the code that ran is the code in the bucket" checkable afterwards.
    _s3 = boto3.client("s3", region_name=REGION)
    Path(LOCAL_PIPELINE_SRC).mkdir(exist_ok=True)
    for fn in ["pipeline_lib.py", "preprocess.py", "train.py", "inference.py"]:
        key = f"{PREFIX}/pipeline_src_v11/{fn}"
        _s3.upload_file(f"src/{fn}", BUCKET, key)
        _s3.download_file(BUCKET, key, str(Path(LOCAL_PIPELINE_SRC, fn)))
    print(f"published + retrieved: {sorted(p.name for p in Path(LOCAL_PIPELINE_SRC).iterdir())}")

    # --- Parameters. InputDataUrl is the one that makes this pipeline triggerable.
    p_input   = ParameterString(name="InputDataUrl",     default_value=DEFAULT_INPUT_URI)
    p_lr      = ParameterFloat(name="LearningRate",      default_value=CHAMPION_HPARAMS["learning_rate"])
    p_leaves  = ParameterInteger(name="MaxLeafNodes",    default_value=CHAMPION_HPARAMS["max_leaf_nodes"])
    p_iter    = ParameterInteger(name="MaxIter",         default_value=CHAMPION_HPARAMS["max_iter"])
    p_minleaf = ParameterInteger(name="MinSamplesLeaf",  default_value=CHAMPION_HPARAMS["min_samples_leaf"])
    p_l2      = ParameterFloat(name="L2Regularization",  default_value=CHAMPION_HPARAMS["l2_regularization"])
    p_gate    = ParameterFloat(name="QualityGateR2Log",  default_value=QUALITY_GATE_R2LOG)

    # --- Step 1: Preprocess. The data mount is a DIRECTORY, not a fixed filename:
    # the uploaded object is named per-event (Listings_manual_<ts>.csv), so
    # preprocess.py v1.1 resolves the newest *.csv inside it. The mount is a
    # /data subdirectory so v1.1's recursive glob cannot wander into the
    # SDK-owned code/ and entrypoint/ channels.
    MOUNT_DATA = "/opt/ml/processing/input/data"
    processor = FrameworkProcessor(
        estimator_cls=SKLearn, framework_version="1.2-1",
        instance_type=PROCESSING_INSTANCE_TYPE, instance_count=1,
        role=role, sagemaker_session=pipeline_session,
        code_location=f"s3://{BUCKET}/{PREFIX}/pipeline_code_v11",
        base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-trig-process")
    proc_args = processor.run(
        code="preprocess.py", source_dir=LOCAL_PIPELINE_SRC,
        inputs=[ProcessingInput(input_name="snapshot", source=p_input, destination=MOUNT_DATA)],
        outputs=[ProcessingOutput(output_name="processed", source="/opt/ml/processing/output",
                                  destination=f"{PIPELINE_ROOT}/processed")],
        arguments=["--input", MOUNT_DATA,
                   "--output-dir", "/opt/ml/processing/output",
                   "--test-size", str(TEST_SIZE), "--random-state", str(RANDOM_STATE)])
    step_process = ProcessingStep(name="PreprocessData", step_args=proc_args)

    # --- Step 2: Train
    estimator = SKLearn(
        entry_point="train.py", source_dir=LOCAL_PIPELINE_SRC, framework_version="1.2-1",
        instance_type=TRAINING_INSTANCE_TYPE, instance_count=1, role=role,
        base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-trig-train",
        sagemaker_session=pipeline_session,
        code_location=f"s3://{BUCKET}/{PREFIX}/pipeline_code_v11",
        hyperparameters={"learning-rate": p_lr, "max-leaf-nodes": p_leaves, "max-iter": p_iter,
                         "min-samples-leaf": p_minleaf, "l2-regularization": p_l2,
                         "gate-r2": p_gate, "random-state": RANDOM_STATE,
                         "team-id": TEAM_ID, "student-id": STUDENT_ID},
        environment={"TEAM_ID": TEAM_ID, "STUDENT_ID": STUDENT_ID, "SEMESTER": SEMESTER},
        metric_definitions=[
            {"Name": "test_mae_log",  "Regex": "test_mae_log: ([0-9\\.]+)"},
            {"Name": "test_rmse_log", "Regex": "test_rmse_log: ([0-9\\.]+)"},
            {"Name": "test_r2_log",   "Regex": "test_r2_log: ([0-9\\.]+)"},
            {"Name": "test_mape",     "Regex": "test_mape: ([0-9\\.]+)"},
        ],
        tags=[{"Key": "Course", "Value": COURSE}, {"Key": "Semester", "Value": SEMESTER},
              {"Key": "Team", "Value": TEAM_ID}, {"Key": "Student", "Value": STUDENT_ID},
              {"Key": "Trigger", "Value": "s3-event"}])
    processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri
    step_train = TrainingStep(
        name="TrainModel", estimator=estimator,
        inputs={"train": sagemaker.inputs.TrainingInput(s3_data=processed_uri, content_type="text/csv"),
                "test":  sagemaker.inputs.TrainingInput(s3_data=processed_uri, content_type="text/csv")})

    # --- Steps 3-4: register behind the same absolute R2_log gate, PendingManualApproval.
    # An automated trigger must not be able to promote anything on its own --
    # the gate admits a challenger to the registry; a human still approves it.
    model = Model(image_uri=estimator.training_image_uri(REGION),
                  model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                  sagemaker_session=pipeline_session, role=role,
                  entry_point="inference.py", source_dir=LOCAL_PIPELINE_SRC)
    step_register = ModelStep(
        name="RegisterModel",
        step_args=model.register(
            content_types=["application/json"], response_types=["application/json"],
            inference_instances=["ml.m5.large"], transform_instances=["ml.m5.large"],
            model_package_group_name=MODEL_PACKAGE_GROUP,
            approval_status="PendingManualApproval"))
    step_condition = ConditionStep(
        name="R2LogQualityGate",
        conditions=[ConditionGreaterThanOrEqualTo(
            left=step_train.properties.FinalMetricDataList["test_r2_log"].Value, right=p_gate)],
        if_steps=[step_register], else_steps=[])

    triggered_pipeline = Pipeline(
        name=TRIGGERED_PIPELINE_NAME,
        parameters=[p_input, p_lr, p_leaves, p_iter, p_minleaf, p_l2, p_gate],
        steps=[step_process, step_train, step_condition],
        sagemaker_session=pipeline_session)
    triggered_pipeline.upsert(
        role_arn=role,
        tags=[{"Key": "Course", "Value": COURSE}, {"Key": "Semester", "Value": SEMESTER},
              {"Key": "Team", "Value": TEAM_ID}, {"Key": "Student", "Value": STUDENT_ID}])

    _desc = sm_client.describe_pipeline(PipelineName=TRIGGERED_PIPELINE_NAME)
    _params = [p["Name"] for p in json.loads(_desc["PipelineDefinition"])["Parameters"]]
    print(f"\nUpserted: {TRIGGERED_PIPELINE_NAME}")
    print(f"  ARN       : {_desc['PipelineArn']}")
    print(f"  parameters: {_params}")
    assert "InputDataUrl" in _params, "InputDataUrl missing — Step 6/Step 7 would fail on start"
    print("  InputDataUrl present — Step 6 and Step 7 can now start this pipeline.")


See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


published + retrieved: ['_repack_model.py', '_repack_script_launcher.sh', 'inference.py', 'pipeline_lib.py', 'preprocess.py', 'train.py']


See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


...

INFO:sagemaker.processing:Uploaded pipeline_src_v11 to s3://nyp-26s1-iti113/iti113/team14/data/airbnb-listings/pipeline_code_v11/iti113-team14-airbnb-price-triggered/code/5b668235709af32de334cfd5ff23b47476294c0ef35f7af92e0bbe9b99cf54c8/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-ap-southeast-1-044528205969/iti113-team14-airbnb-price-triggered/code/0d453f34961f367d397854869f8b2673fec384779e12ff533eda22686866dfa6/runproc.sh


INFO:sagemaker.processing:Uploaded pipeline_src_v11 to s3://nyp-26s1-iti113/iti113/team14/data/airbnb-listings/pipeline_code_v11/iti113-team14-airbnb-price-triggered/code/5b668235709af32de334cfd5ff23b47476294c0ef35f7af92e0bbe9b99cf54c8/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-ap-southeast-1-044528205969/iti113-team14-airbnb-price-triggered/code/0d453f34961f367d397854869f8b2673fec384779e12ff533eda22686866dfa6/runproc.sh



Upserted: iti113-team14-airbnb-price-triggered
  ARN       : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team14-airbnb-price-triggered
  parameters: ['InputDataUrl', 'LearningRate', 'MaxLeafNodes', 'MaxIter', 'MinSamplesLeaf', 'L2Regularization', 'QualityGateR2Log']
  InputDataUrl present — Step 6 and Step 7 can now start this pipeline.


## 6. Manual Pipeline Test — Validating Before Automation

The sequence outlined in the reference notebook highlights a critical operational rule: **never automate a pipeline that you haven't first executed manually.** From a systems perspective, a failed trigger and a failed pipeline look exactly the same. By successfully running the pipeline by hand first, we ensure that when the automation layer is finally introduced, any future failures can be isolated strictly to the trigger mechanism itself.

This manual execution also serves as a **parameter-override demonstration**. By temporarily setting `MaxIter=40` (instead of the champion model's 300), we execute a rapid smoke test of the entire execution path—covering `InputDataUrl` resolution, data cleaning, train/test splitting, model fitting, and the quality gate—without burning the compute required for a full production-grade run. 

Crucially, by design, manual runs terminate exactly at the quality gate. **Model registration and promotion are strictly reserved for the automated, governed trigger path.** This hard boundary ensures that ad-hoc, hand-started experiments can never accidentally overwrite the production `@champion` alias.

In [12]:
manual_exec = run_retraining_pipeline(
    SNAPSHOT_APRIL, event_id=f"manual-test-{int(time.time())}",
    params={"MaxIter": 40})                       # smoke-strength override
print()
print_step_table(manual_exec)
assert manual_exec["status"] == "Succeeded", "fix the pipeline before wiring the trigger"

mr = json.loads((Path(manual_exec["run_dir"]) / "artifacts" / "evaluation_report.json").read_text())
print(f"\nSmoke model (MaxIter=40): test_r2_log {mr['metrics']['test_r2_log']} "
      f"| gate {'PASSED' if mr['quality_gate']['passed'] else 'FAILED'} "
      f"(threshold {QUALITY_GATE_R2LOG})")
print("Manual path validated — registration/promotion deliberately NOT performed here.")

Execution: iti113-team14-airbnb-price-triggered/manual-test-1786893788
PipelineParameters: InputDataUrl=Listings_2021-04.csv, MaxIter=40, QualityGateR2Log=0.65
  PreprocessData   Executing ...

  PreprocessData   Succeeded  (9.9s)
  TrainModel       Executing ...

  TrainModel       Succeeded  (48.0s)

Pipeline status: Succeeded   (manual-test-1786893788)
  - PreprocessData   Succeeded     9.9s
  - TrainModel       Succeeded    48.0s

Smoke model (MaxIter=40): test_r2_log 0.8433 | gate PASSED (threshold 0.65)
Manual path validated — registration/promotion deliberately NOT performed here.


In [13]:
if LOCAL_MODE:
    print("[SKIP] LOCAL_MODE — on Studio, the manual test uploads to the NON-watched prefix and")
    print("       starts the triggered pipeline explicitly with InputDataUrl:")
    print(f"         s3://{BUCKET}/{MANUAL_TEST_PREFIX}/Listings_manual_<ts>.csv")
else:
    s3 = boto3.client("s3")
    manual_key = f"{MANUAL_TEST_PREFIX}/Listings_manual_{int(time.time())}.csv"
    s3.upload_file(str(SNAPSHOT_APRIL), BUCKET, manual_key)
    manual_uri = f"s3://{BUCKET}/{manual_key}"
    print(f"Uploaded manual test file (non-watched prefix): {manual_uri}")
    resp = sm_client.start_pipeline_execution(
        PipelineName=TRIGGERED_PIPELINE_NAME,
        PipelineExecutionDisplayName=f"manual-triggered-test-{int(time.time())}",
        PipelineParameters=[
            {"Name": "InputDataUrl", "Value": manual_uri},
            {"Name": "MaxIter", "Value": "40"},
            {"Name": "QualityGateR2Log", "Value": str(QUALITY_GATE_R2LOG)},
        ])
    MANUAL_EXECUTION_ARN = resp["PipelineExecutionArn"]
    print(f"Manual pipeline execution started: {MANUAL_EXECUTION_ARN}")

Uploaded manual test file (non-watched prefix): s3://nyp-26s1-iti113/iti113/team14/manual-input/Listings_manual_1786893846.csv


Manual pipeline execution started: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team14-airbnb-price-triggered/execution/k086ocgkdudq


### 6.1 Polling Helper for Managed Executions

We retain the core logic of the reference notebook's `poll_pipeline_execution` utility. Its purpose is straightforward: continuously query the pipeline's state, catalog the progress of individual steps, and explicitly surface any `FailureReason` until the execution reaches a terminal state (either completion or failure). 

While our local retraining runner via its step-status table naturally fulfills this role during local execution, this helper function is critical when operating within SageMaker Studio. It is the exact cell you must execute immediately following any `start_pipeline_execution` call to actively monitor the managed cloud infrastructure.

In [14]:
def poll_pipeline_execution(execution_arn, sleep_seconds=30):
    """Poll a SageMaker Pipeline execution until terminal, printing step statuses
    and failure reasons (Studio only)."""
    import boto3
    sm = boto3.client("sagemaker", region_name=REGION)
    terminal = ("Succeeded", "Failed", "Stopped")
    while True:
        desc = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)
        status = desc["PipelineExecutionStatus"]
        print("=" * 70)
        print(f"Pipeline status: {status}  ({desc.get('PipelineExecutionDisplayName')})")
        steps = sm.list_pipeline_execution_steps(
            PipelineExecutionArn=execution_arn).get("PipelineExecutionSteps", [])
        for step in reversed(steps):
            print(f"  - {step.get('StepName'):<20} {step.get('StepStatus')}")
            if step.get("FailureReason"):
                print(f"      FailureReason: {step['FailureReason']}")
        if status in terminal:
            print(f"\nFinal status: {status}")
            return status
        time.sleep(sleep_seconds)

print("poll_pipeline_execution() defined"
      + (" — Studio-side; the local runner's step table is its stand-in here."
         if LOCAL_MODE else ".")
      )
if not LOCAL_MODE:
    poll_pipeline_execution(MANUAL_EXECUTION_ARN)

poll_pipeline_execution() defined.
Pipeline status: Executing  (manual-triggered-test-1786893846)


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Executing


...

Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Executing  (manual-triggered-test-1786893846)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Succeeded  (manual-triggered-test-1786893846)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Succeeded
  - RegisterModel-RegisterModel Succeeded

Final status: Succeeded


## 7. The Trigger — Development Path: The Notebook Monitoring Loop

At its core, this deliverable relies on a straightforward mechanism: **a polling loop that monitors a specific folder and triggers the pipeline the moment a genuinely new file arrives.** During development, it is crucial to have a trigger whose entire decision-making process is transparent and visible directly within a notebook cell. Every concept demonstrated in this local loop has a direct, production-grade counterpart that will take over in Step 10:

| Local Monitoring Loop *(Executed Here)* | Production Counterpart *(Step 10)* |
| :--- | :--- |
| Periodically `list` the folder and compare against a ledger. | S3 → EventBridge `ObjectCreated` events (push-based, no polling required). |
| Idempotency ledger (`processed_uploads.json` tracked by object and size). | Deterministic execution naming and in-progress checks via Lambda (handles EventBridge's *at-least-once* delivery). |
| Starts pipeline setting `InputDataUrl` to the new object. | Lambda invokes `start_pipeline_execution` using the exact same parameter. |
| Local step-status table and dedicated log files. | SageMaker's `list_pipeline_execution_steps` API and CloudWatch log streams. |
| Failed uploads moved to `quarantine/` alongside retained logs. | Identical quarantine move, plus an SQS Dead Letter Queue (DLQ) for Lambda-level faults. |
| Gracefully exits after two idle cycles. | Always-on, long-lived EventBridge rule (no exit required). |

The inherent weaknesses of this local loop are exactly the point: it only functions while a human keeps the cell running, it introduces polling latency, and it burns through active notebook compute. These three critical gaps are precisely why we transition to an automated EventBridge/Lambda architecture for production.

In [15]:
def list_new_uploads(state):
    """Everything in the watched folder not yet in the idempotency ledger.
    Returns (display_name, locator, ledger_key) triples; locator is a local path
    in LOCAL_MODE or an s3:// URI on Studio."""
    found = []
    if LOCAL_MODE:
        for p in sorted(WATCH_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime):
            found.append((p.name, p, upload_key(p)))
    else:
        resp = boto3.client("s3").list_objects_v2(Bucket=BUCKET, Prefix=f"{TRIGGER_PREFIX}/")
        for o in resp.get("Contents", []):
            if o["Key"].endswith(".csv"):
                found.append((o["Key"].rsplit("/", 1)[-1],
                              f"s3://{BUCKET}/{o['Key']}",
                              f"{o['Key']}|{o['Size']}"))
    return [f for f in found if f[2] not in state]

def watch_and_trigger(poll_seconds=4, max_cycles=8, params=None):
    """The development-grade event loop: poll -> detect -> retrain -> compare -> register.
    On Studio the same loop starts the managed pipeline (InputDataUrl = the S3 object)
    and polls it; gate + registration then follow the managed path of Notebook 03 SS7.3."""
    state, idle, handled = load_state(), 0, 0
    where = WATCH_DIR if LOCAL_MODE else f"s3://{BUCKET}/{TRIGGER_PREFIX}"
    print(f"Watching {where}/ every {poll_seconds}s "
          f"(ledger: {len(state)} uploads already processed)\n")
    for cycle in range(1, max_cycles + 1):
        ts = datetime.now().strftime("%H:%M:%S")
        fresh = list_new_uploads(state)
        if not fresh:
            idle += 1
            print(f"[{ts}] cycle {cycle}: no new uploads ({len(state)} in ledger)")
            if idle >= 2:
                print(f"\nQueue drained — exiting after {idle} idle cycles. "
                      f"Handled this session: {handled}.")
                break
            time.sleep(poll_seconds)
            continue
        idle = 0
        for name, locator, key in fresh:
            event_id = (f"evt-{datetime.now():%Y%m%d-%H%M%S}-"
                        f"{hashlib.sha1(key.encode()).hexdigest()[:6]}")
            print(f"[{ts}] cycle {cycle}: NEW UPLOAD -> {name}   (event {event_id})")
            t0 = time.time()
            record = {"event_id": event_id, "file": name, "trigger": "notebook-loop"}
            if LOCAL_MODE:
                execution = run_retraining_pipeline(locator, event_id, params)
                record.update(status=execution["status"],
                              duration_s=round(time.time() - t0, 1))
                if execution["status"] == "Succeeded":
                    outcome = evaluate_and_register(execution)
                    record.update(model_version=outcome["model_version"],
                                  promoted=outcome["promoted"])
                    state[key] = {"event_id": event_id, "status": "Succeeded",
                                  "model_version": outcome["model_version"],
                                  "promoted": outcome["promoted"],
                                  "at": datetime.now(timezone.utc).isoformat()}
                else:
                    qpath = QUARANTINE_DIR / name
                    shutil.move(str(locator), qpath)
                    record.update(failure=execution["steps"][-1]["FailureReason"],
                                  quarantined=str(qpath), run_dir=execution["run_dir"])
                    print(f"  Upload QUARANTINED -> {qpath}")
                    print(f"  Logs preserved for forensics: {execution['run_dir']}/logs/")
                    state[key] = {"event_id": event_id, "status": "Failed",
                                  "quarantined": str(qpath), "run_dir": execution["run_dir"],
                                  "at": datetime.now(timezone.utc).isoformat()}
            else:
                resp = sm_client.start_pipeline_execution(
                    PipelineName=TRIGGERED_PIPELINE_NAME,
                    PipelineExecutionDisplayName=event_id,
                    ClientRequestToken=("nb" + hashlib.sha1(key.encode()).hexdigest()[:30]),
                    PipelineParameters=[{"Name": "InputDataUrl", "Value": locator},
                                        {"Name": "QualityGateR2Log",
                                         "Value": str(QUALITY_GATE_R2LOG)}])
                status = poll_pipeline_execution(resp["PipelineExecutionArn"])
                record.update(status=status, duration_s=round(time.time() - t0, 1),
                              execution_arn=resp["PipelineExecutionArn"])
                state[key] = {"event_id": event_id, "status": status,
                              "at": datetime.now(timezone.utc).isoformat()}
                if status != "Succeeded":
                    # Mirror the LOCAL_MODE branch: name the failing step and move the
                    # offending object out of the watched prefix, so a redeployed
                    # EventBridge rule cannot re-fire on it.
                    _steps = sm_client.list_pipeline_execution_steps(
                        PipelineExecutionArn=resp["PipelineExecutionArn"])["PipelineExecutionSteps"]
                    _bad = next((s for s in _steps if s["StepStatus"] == "Failed"), None)
                    if _bad:
                        record.update(failed_step=_bad["StepName"],
                                      failure=_bad.get("FailureReason", ""))
                        # Persist to the LEDGER too, not just the in-memory EVENT_LOG,
                        # so Step 9 can reconstruct the incident after a kernel restart.
                        state[key]["failed_step"] = _bad["StepName"]
                        state[key]["failure"] = _bad.get("FailureReason", "")
                    state[key]["execution_arn"] = resp["PipelineExecutionArn"]
                    _s3q = boto3.client("s3", region_name=REGION)
                    _src_key = locator.split(f"s3://{BUCKET}/", 1)[-1]
                    _q_key = f"{QUARANTINE_PREFIX}/{name}"
                    _s3q.copy_object(Bucket=BUCKET, Key=_q_key,
                                     CopySource={"Bucket": BUCKET, "Key": _src_key})
                    _s3q.delete_object(Bucket=BUCKET, Key=_src_key)
                    record.update(quarantined=f"s3://{BUCKET}/{_q_key}")
                    state[key]["quarantined"] = f"s3://{BUCKET}/{_q_key}"
                    print(f"  Upload QUARANTINED -> s3://{BUCKET}/{_q_key}")
                    print(f"  Failing step: {record.get('failed_step', 'unknown')}")
                else:
                    print("  Managed run complete — gate + registration follow the")
                    print("  Notebook 03 SS7.3 post-pipeline path (registry PendingManualApproval).")
            save_state(state)
            EVENT_LOG.append(record)
            handled += 1
            print()
    return handled

print("watch_and_trigger() ready.")

watch_and_trigger() ready.


### 7.1 Arm the trigger — upload the April snapshot

*This* is the moment the deliverable describes: a CSV lands in the controlled folder. Locally that's a copy into the watched directory; on Studio it's the guarded `s3.upload_file` into `trigger/input/` (which, with Step 10's infrastructure live, would fire EventBridge without any loop at all).

In [16]:
dst = WATCH_DIR / SNAPSHOT_APRIL.name
shutil.copy2(SNAPSHOT_APRIL, dst)
print(f"UPLOADED: {dst}")
print(f"          ({dst.stat().st_size/1e6:.0f} MB, key {upload_key(dst)})")
print()
print("Expected chain: watched folder -> monitoring loop -> "
      f"{TRIGGERED_PIPELINE_NAME} -> gate -> challenger-vs-champion -> registry")

if not LOCAL_MODE:
    trigger_key = f"{TRIGGER_PREFIX}/{SNAPSHOT_APRIL.name}"
    boto3.client("s3").upload_file(str(SNAPSHOT_APRIL), BUCKET, trigger_key)
    print(f"\n(Studio) Uploaded s3://{BUCKET}/{trigger_key} — with Step 10 deployed, "
          f"EventBridge -> Lambda would start the pipeline with no loop running.")

UPLOADED: s3_mirror/iti113/team14/trigger/input/Listings_2021-04.csv
          (146 MB, key Listings_2021-04.csv|146046994)

Expected chain: watched folder -> monitoring loop -> iti113-team14-airbnb-price-triggered -> gate -> challenger-vs-champion -> registry



(Studio) Uploaded s3://nyp-26s1-iti113/iti113/team14/trigger/input/Listings_2021-04.csv — with Step 10 deployed, EventBridge -> Lambda would start the pipeline with no loop running.


In [17]:
handled = watch_and_trigger(poll_seconds=4, max_cycles=8)

Watching s3://nyp-26s1-iti113/iti113/team14/trigger/input/ every 4s (ledger: 2 uploads already processed)

[15:35:42] cycle 1: NEW UPLOAD -> Listings_live_1786722834.csv   (event evt-20260816-153542-3ffef0)


Pipeline status: Executing  (evt-20260816-153542-3ffef0)


Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Executing


Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Executing


Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Executing


...

Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Executing  (evt-20260816-153542-3ffef0)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Executing


Pipeline status: Succeeded  (evt-20260816-153542-3ffef0)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Succeeded
  - RegisterModel-RegisterModel Succeeded

Final status: Succeeded
  Managed run complete — gate + registration follow the
  Notebook 03 SS7.3 post-pipeline path (registry PendingManualApproval).

[15:47:47] cycle 2: no new uploads (3 in ledger)


[15:47:51] cycle 3: no new uploads (3 in ledger)

Queue drained — exiting after 2 idle cycles. Handled this session: 1.


**Read that output top to bottom and the whole deliverable is in it:** the loop detected the upload on its first cycle, started the *same* validated pipeline with `InputDataUrl` pointing at the new snapshot, both steps succeeded, the gate held, the challenger was compared against the champion **by written policy**, registered as **v2** with full lineage tags, and — having passed both the absolute floor and the non-inferiority test — was **promoted**: `@champion` now points at a model trained on April data, and the idempotency ledger remembers the upload. Two idle cycles later the loop bowed out, because a notebook loop should know it's the understudy.

### 7.2 The idempotency guarantee, demonstrated

Duplicate delivery isn't hypothetical — pollers re-list every cycle and EventBridge is at-least-once. Re-running the loop with the same folder contents must therefore be a **no-op**: the April upload sits right there, and the ledger refuses it.

In [18]:
_ = watch_and_trigger(poll_seconds=2, max_cycles=2)
print(f"\nCurrent champion after the event: v{CHAMPION['version']} "
      f"(test_r2_log {CHAMPION['metrics']['test_r2_log']})")

Watching s3://nyp-26s1-iti113/iti113/team14/trigger/input/ every 2s (ledger: 3 uploads already processed)

[15:47:51] cycle 1: no new uploads (3 in ledger)


[15:47:53] cycle 2: no new uploads (3 in ledger)

Queue drained — exiting after 2 idle cycles. Handled this session: 0.

Current champion after the event: v7 (test_r2_log 0.8708)


## 8. Failure Drill — a poison upload

Automation that only ever sees good inputs is untested automation. The most common real-world trigger failure is **schema drift** — an upstream producer changes their export and the file that lands is subtly wrong. We manufacture exactly that: a May "snapshot" missing the `amenities` column, dropped straight into the watched folder. The contract says what must happen: the pipeline **fails fast at the step that notices** (training's serving-contract check), the failure reason is captured, the upload is **quarantined instead of retried**, and the ledger records the corpse so it can never re-trigger.

In [19]:
# --- Step 8 Failure drill: a poison upload (schema drift), sent down the SAME path as a good one
# The ledger is keyed on filename|size and PERSISTS ACROSS KERNEL RESTARTS, so re-running
# this cell regenerates a byte-identical file whose key is already recorded — the watcher
# will (correctly) refuse to re-trigger. Set FORCE_NEW_DRILL to run a genuinely new one.
FORCE_NEW_DRILL = False       # True -> unique filename -> a fresh, billed, failing execution

bad = pd.read_csv("Listings.csv" if os.path.exists("Listings.csv")
                  else "/mnt/user-data/uploads/Listings.csv",
                  nrows=800, encoding="utf-8", encoding_errors="replace", low_memory=False)
bad = bad.drop(columns=["amenities"])                 # upstream schema drift

_stem = f"Listings_2021-05_schema_drift{'_' + str(int(time.time())) if FORCE_NEW_DRILL else ''}"
BAD_SNAPSHOT = WATCH_DIR / f"{_stem}.csv"
bad.to_csv(BAD_SNAPSHOT, index=False)
print(f"Wrote poison snapshot: {BAD_SNAPSHOT.name} — {len(bad)} rows, 'amenities' column missing")

_state = load_state()
_key = upload_key(BAD_SNAPSHOT)
_prior = _state.get(_key)

if _prior and not FORCE_NEW_DRILL:
    print(f"\nLedger already holds {_key!r} — this drill ran on "
          f"{_prior.get('at', 'an earlier session')}:")
    for f in ("event_id", "status", "failed_step", "quarantined"):
        if _prior.get(f):
            print(f"    {f:12s} {_prior[f]}")
    print("\nThe watcher will skip it: one execution per object version is the guarantee")
    print("Step 7 exists to demonstrate. Step 9 reads this record from the ledger, so it still works.")
    print("Set FORCE_NEW_DRILL = True to run a fresh failing execution under a new name.")
else:
    # On Studio the watcher lists S3, not the local mirror, so the drill only fires if the
    # object actually lands in the watched PREFIX.
    if not LOCAL_MODE:
        bad_key = f"{TRIGGER_PREFIX}/{BAD_SNAPSHOT.name}"
        boto3.client("s3", region_name=REGION).upload_file(str(BAD_SNAPSHOT), BUCKET, bad_key)
        print(f"UPLOADED (poison): s3://{BUCKET}/{bad_key}\n")
    else:
        print(f"UPLOADED (poison): {BAD_SNAPSHOT}\n")

    _ = watch_and_trigger(poll_seconds=2, max_cycles=4)


Wrote poison snapshot: Listings_2021-05_schema_drift.csv — 800 rows, 'amenities' column missing
UPLOADED (poison): s3://nyp-26s1-iti113/iti113/team14/trigger/input/Listings_2021-05_schema_drift.csv

Watching s3://nyp-26s1-iti113/iti113/team14/trigger/input/ every 2s (ledger: 3 uploads already processed)

[15:47:53] cycle 1: no new uploads (3 in ledger)


[15:47:55] cycle 2: no new uploads (3 in ledger)

Queue drained — exiting after 2 idle cycles. Handled this session: 0.


## 9. Monitoring & Troubleshooting — Reading the Pipeline's Vital Signs

Effective troubleshooting always follows an **outside-in** methodology: start with the overall execution status → identify the specific failed step → check its `FailureReason` → dive into that step's log stream → locate the exact traceback → and finally, classify the root cause (is it a *data*, *code*, or *infrastructure* issue?). 

Every diagnostic signal along this path is fully accessible in both our local simulation and the live production environment:

| Diagnostic Signal | Local Environment *(Executed Here)* | Production Environment *(AWS)* |
| :--- | :--- | :--- |
| **Execution & Step Status** | The local runner's step-status table. | SageMaker APIs: `describe_pipeline_execution` and `list_pipeline_execution_steps`. |
| **Primary Failure Reason** | Extracted via `_tail_error()` from the local log. | The native `FailureReason` field on the job or step. |
| **Processing & Training Logs** | Local files in `pipeline_runs/<event>/logs/<Step>.log`. | CloudWatch log groups: `/aws/sagemaker/ProcessingJobs` and `/aws/sagemaker/TrainingJobs`. |
| **Trigger Mechanism Logs** | The printed output of the notebook cell. | CloudWatch log group: `/aws/lambda/iti113-team14-airbnb-trigger`. |
| **System Event History** | `processed_uploads.json` ledger + local `EVENT_LOG`. | EventBridge rule metrics combined with the SageMaker execution list. |

**Case Study: The May Data Failure**
Our simulated failure with the May dataset serves as a perfect example of this system in action. Notice how the architecture's *step isolation* points us toward the answer before we even open a single log file. 

The `PreprocessData` step **succeeded** (because the initial data cleaning phase doesn't heavily interact with the `amenities` column). However, the `TrainModel` step **failed** the exact moment it tried to enforce the serving contract via `df[SERVING_COLUMNS]`. This is the system working exactly as designed. Because `SERVING_COLUMNS` defines the strict schema contract, the pipeline correctly halts training rather than producing a model that would eventually crash the live serving endpoint due to missing data.

In [20]:
# --- Forensics on the quarantined May upload (the CloudWatch workflow, executed)
# EVENT_LOG is in-memory and resets with the kernel; the ledger on disk is durable.
# Prefer the live record, fall back to the ledger, so Step 9 survives a restart.
failed = next((r for r in EVENT_LOG if r.get("status") == "Failed"), None)
_source = "EVENT_LOG (this session)"

if failed is None:
    _st = load_state()
    _k, _rec = next(((k, v) for k, v in _st.items() if v.get("status") == "Failed"),
                    (None, None))
    if _rec:
        failed = {"status": "Failed", "event_id": _rec.get("event_id"),
                  "file": _k.split("|")[0], "failed_step": _rec.get("failed_step"),
                  "failure": _rec.get("failure"), "quarantined": _rec.get("quarantined"),
                  "execution_arn": _rec.get("execution_arn")}
        _source = f"idempotency ledger ({STATE_FILE.name}, recorded {_rec.get('at', 'earlier')})"

if failed is None:
    raise SystemExit(
        "No Failed upload in EVENT_LOG or the ledger — Step 8's drill has never completed.\n"
        "Run Step 8. If it reports the file is already in the ledger, that record IS the\n"
        "evidence and this cell will read it; set FORCE_NEW_DRILL=True only if you want\n"
        "a fresh failing execution.")

print(f"source: {_source}\n")
print(f"1) Execution status  : {failed['status']}   (event {failed['event_id']}, "
      f"file {failed['file']})")
print(f"2) Failing step      : {failed.get('failed_step') or 'see step list in Step 9.1'}")
print(f"3) FailureReason     : {failed.get('failure') or '(pull live via Step 9.1)'}")

if failed.get("run_dir"):                       # local simulation: read the captured log
    log_path = Path(failed["run_dir"]) / "logs" / f"{failed.get('failed_step', 'TrainModel')}.log"
    if not log_path.exists():
        cands = sorted(Path(failed["run_dir"], "logs").glob("*.log"))
        log_path = cands[-1] if cands else None
    if log_path:
        lines = log_path.read_text(errors="replace").splitlines()
        tb = [i for i, l in enumerate(lines) if l.startswith("Traceback")]
        print(f"4) Log stream tail   : {log_path}")
        for l in lines[max(tb) if tb else 0:]:
            print(f"     {l}")
else:                                           # managed run: the traceback is in CloudWatch
    _arn = failed.get("execution_arn")
    print(f"4) Log stream tail   : CloudWatch — see Step 9.1, execution "
          f"{_arn.rsplit('/', 1)[-1] if _arn else '(resolved live in Step 9.1)'}")
if failed.get("quarantined"):
    print(f"   Quarantined to    : {failed['quarantined']}")

print("\n5) Classification    : DATA failure (upstream schema drift) — not code, not infra.")
print("   The upload is quarantined; the ledger blocks re-triggering; the producer gets the")
print("   traceback verbatim. Fix at the source, re-upload under a new snapshot name.")
print("   Cheap prevention (added to the Lambda in Step 10): sniff the CSV header against")
print(f"   SERVING_COLUMNS before ever starting an execution — {len(SERVING_COLUMNS)} required fields.")


source: idempotency ledger (processed_uploads.json, recorded 2026-08-14T10:32:45.658045+00:00)

1) Execution status  : Failed   (event evt-20260814-102713-340fea, file iti113/team14/trigger/input/Listings_2021-05_schema_drift.csv)
2) Failing step      : TrainModel
3) FailureReason     : (pull live via Step 9.1)
4) Log stream tail   : CloudWatch — see Step 9.1, execution (resolved live in Step 9.1)
   Quarantined to    : s3://nyp-26s1-iti113/iti113/team14/trigger/quarantine/Listings_2021-05_schema_drift.csv

5) Classification    : DATA failure (upstream schema drift) — not code, not infra.
   The upload is quarantined; the ledger blocks re-triggering; the producer gets the
   traceback verbatim. Fix at the source, re-upload under a new snapshot name.
   Cheap prevention (added to the Lambda in Step 10): sniff the CSV header against
   SERVING_COLUMNS before ever starting an execution — 27 required fields.


### 9.1 Performing Forensics on Real AWS Services

The cells below provide the production-grade counterparts to steps 1–4. They are designed to be executed verbatim directly within SageMaker Studio following any pipeline failure. They trace the exact same diagnostic path we established locally: pulling the latest execution → extracting step-level failure reasons → retrieving the specific `FailureReason` from the training job → querying the tail of the CloudWatch logs → and finally, inspecting the Lambda function's log group (which is essential for catching *trigger*-layer failures, where a file is uploaded but a pipeline execution never actually starts).

In [21]:
if LOCAL_MODE:
    print("[SKIP] LOCAL_MODE — the boto3 forensics below run on Studio; Step 9's local walk-through is their exact stand-in.")
else:
    logs = boto3.client("logs", region_name=REGION)

    # 1-2) Most recent FAILED triggered execution (not merely the most recent one --
    # a later successful run would otherwise hide the failure we came to investigate)
    summaries = sm_client.list_pipeline_executions(
        PipelineName=TRIGGERED_PIPELINE_NAME, SortBy="CreationTime",
        SortOrder="Descending", MaxResults=20)["PipelineExecutionSummaries"]
    ex = next((e for e in summaries if e["PipelineExecutionStatus"] == "Failed"), None)
    if ex is None:
        raise SystemExit(f"No failed execution of {TRIGGERED_PIPELINE_NAME} in the last "
                         f"{len(summaries)} — run Step 8's poison drill first.")
    print(f"execution: {ex['PipelineExecutionArn'].rsplit('/', 1)[-1]}  "
          f"({ex['PipelineExecutionStatus']})\n")

    steps = sm_client.list_pipeline_execution_steps(
        PipelineExecutionArn=ex["PipelineExecutionArn"])["PipelineExecutionSteps"]
    failed_step, job_name, job_kind = None, None, None
    for s in reversed(steps):
        print(f"{s['StepName']:<20} {s['StepStatus']:<10} {s.get('FailureReason', '')}")
        if s["StepStatus"] == "Failed":
            failed_step = s["StepName"]
            md = s.get("Metadata", {})
            # Pipeline-generated jobs are named pipelines-<execId>-<Step>-<rand>, NOT
            # after base_job_name -- so resolve the ARN from step metadata rather than
            # searching by name substring, which never matches.
            for kind, group in (("TrainingJob", "/aws/sagemaker/TrainingJobs"),
                                ("ProcessingJob", "/aws/sagemaker/ProcessingJobs")):
                arn = md.get(kind, {}).get("Arn")
                if arn:
                    job_name, job_kind, log_group = arn.rsplit("/", 1)[-1], kind, group
                    break

    if not job_name:
        raise SystemExit(f"Step {failed_step} failed before any job was created — "
                         f"the reason above is the whole story (definition or IAM).")

    # 3) The failed job's own FailureReason
    if job_kind == "TrainingJob":
        desc = sm_client.describe_training_job(TrainingJobName=job_name)
        status = desc["TrainingJobStatus"]
    else:
        desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
        status = desc["ProcessingJobStatus"]
    print(f"\nfailing step  : {failed_step}  ({job_kind})")
    print(f"{job_name}: {status} | {desc.get('FailureReason', 'no failure')}")

    # 4) CloudWatch tail for that job (stream prefix = job name)
    streams = logs.describe_log_streams(logGroupName=log_group,
                                        logStreamNamePrefix=job_name).get("logStreams", [])
    if not streams:
        print(f"\nNo streams under {log_group}/{job_name} — the container never started.")
    else:
        print(f"\n--- {log_group}/{streams[-1]['logStreamName']} ---")
        for e in logs.get_log_events(logGroupName=log_group,
                                     logStreamName=streams[-1]["logStreamName"],
                                     startFromHead=True, limit=40)["events"]:
            print("   ", e["message"].rstrip())


execution: rq848c2jvdmu  (Failed)

PreprocessData       Succeeded  
TrainModel           Failed     ClientError: AlgorithmError: framework error: 
Traceback (most recent call last):
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py", line 84, in train
    entrypoint()
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 39, in main
    train(environment.Environment())
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 31, in train
    entry_point.run(uri=training_environment.module_dir,
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/entry_point.py", line 108, in run
    return runner.get(runner_type, user_entry_point, args, env_vars, extra_opts).run(
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", line 424, in run
    process = check_error(
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", 

    /miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
    2026-08-14 10:32:13,536 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
    2026-08-14 10:32:13,541 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
    2026-08-14 10:32:13,544 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
    2026-08-14 10:32:13,562 sagemaker_sklearn_container.training INFO     Invoking user training script.
    2026-08-14 10:32:13,827 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
    2026-08-14 10:32:13,832 sagemaker-training-toolkit INFO     No Neurons detected (normal if no

### 9.2 The Triage Runbook

Compiled from the reference notebook's hard-learned lessons and our own project drills, here is the practical troubleshooting matrix that on-call engineers actually need:

| Symptom | Initial Check | Common Cause | Resolution |
| :--- | :--- | :--- | :--- |
| `S3UploadFailedError: AccessDenied` during pipeline upsert | Check the SDK upload path in the error message. | Scripts are being staged **outside the designated team prefix**. | Use `PipelineSession(default_bucket=BUCKET, default_bucket_prefix=<team prefix>)`. |
| `TypeError: unexpected keyword 'code_location'` | Check the `SKLearnProcessor(...)` instantiation. | This specific processor does not support the `code_location` argument. | Rely on the `PipelineSession` prefix to handle script staging automatically. |
| `FileNotFoundError: model_pipeline.joblib` occurring in downstream steps | Inspect the first 20 lines of the step's log. | The model channel delivers a compressed **`model.tar.gz`**, not the raw file. | Extract the tarball before attempting to load the model (our pipeline avoids this hop by reading train-time metrics directly). |
| Pipeline status is **Succeeded** but no new version appears in the registry | Compare the `TrainModel` metric outputs against the gate parameter. | The model's performance fell below the quality gate threshold, so registration was skipped *by design*. | Validate the regex-captured metric; either retune the model or accept the rejection. |
| `KeyError: [...] not in index` during `TrainModel` | Review the `FailureReason` followed by the traceback. | The newly uploaded snapshot suffers from **schema drift**. | Quarantine the file, enforce the producer data contract, and implement a header pre-check in the Lambda function *(as drilled in Step 8)*. |
| The exact same file triggers repeated retrains | Check the idempotency ledger or the execution list. | Duplicate event delivery (e.g., from at-least-once event buses or re-listing polling loops). | Implement a local idempotency ledger or enforce a deterministic `ClientRequestToken` within the Lambda. |
| File is uploaded, but **no pipeline execution starts** | Inspect the Lambda log group first, then the EventBridge rule metrics. | EventBridge notifications are disabled on the bucket, the rule's path prefix doesn't match, or the Lambda is missing the `sagemaker:StartPipelineExecution` IAM permission. | Walk through the one-time setup checklist from Step 10, top to bottom. |
| Lambda functions time out or throttle during a burst of uploads | Review the Lambda duration and throttling metrics. | The function is hitting the default 3-second timeout limit, or concurrency is unbounded. | Increase the timeout setting (e.g., to ~30 seconds), cap reserved concurrency at 1–2, and configure a DLQ for persistent failures. |

A DLQ stands for Dead Letter Queue. In software engineering and cloud architecture, it is a dedicated safety-net queue where a system sends messages or events that it repeatedly fails to process.

### 10.2 The live end-to-end trigger test *(Studio)*

The reference notebook's final act, adapted: upload a snapshot to the **watched** prefix and touch nothing else. EventBridge and the Lambda need a short moment; then the newest execution is polled to completion. If its steps go green, the entire chain — bucket event → rule → handler → parameterized pipeline → gate → registry — worked with zero human participation past the upload.

In [27]:
if LOCAL_MODE:
    print("[SKIP] LOCAL_MODE — the live chain test runs on Studio; Step 7's loop event was its executed local equivalent.")
else:
    live_key = f"{TRIGGER_PREFIX}/Listings_live_{int(time.time())}.csv"
    boto3.client("s3").upload_file(str(SNAPSHOT_APRIL), BUCKET, live_key)
    print(f"Uploaded s3://{BUCKET}/{live_key}")
    print("Chain: S3 -> EventBridge -> Lambda -> SageMaker Pipeline\n")
    time.sleep(10)                                     # let the event propagate
    latest = sm_client.list_pipeline_executions(
        PipelineName=TRIGGERED_PIPELINE_NAME, SortBy="CreationTime",
        SortOrder="Descending", MaxResults=1)["PipelineExecutionSummaries"]
    if not latest:
        raise RuntimeError("No execution appeared — start forensics at the Lambda log group (Step 9.1).")
    poll_pipeline_execution(latest[0]["PipelineExecutionArn"])

Uploaded s3://nyp-26s1-iti113/iti113/team14/trigger/input/Listings_live_1786895276.csv
Chain: S3 -> EventBridge -> Lambda -> SageMaker Pipeline



Pipeline status: Succeeded  (evt-20260816-153542-3ffef0)
  - PreprocessData       Succeeded
  - TrainModel           Succeeded
  - R2LogQualityGate     Succeeded
  - RegisterModel-RepackModel-0 Succeeded
  - RegisterModel-RegisterModel Succeeded

Final status: Succeeded


## 11. Where the Trigger Fits — CI/CD, the Registry Junction, and the Dashboards

Notebook 03 built a pipeline with **one** trigger: an engineer merging code. This notebook added the **second**: data arriving. Both roads lead through the same junction — the Model Registry — and out the same governed exit:

```text
  CODE changes                                   DATA changes
  PR -> CI (ruff, pytest incl. the               new snapshot ->
  train-serve consistency golden test)           trigger/input/
        │ merge                                        │ S3 event
        ▼                                              ▼
  GitHub Actions (OIDC) ──upsert+start──►  ◄──start── EventBridge -> Lambda
                        THE SAME PIPELINE
                 Process → Train → R²(log) gate
                              │ pass
                              ▼
              MODEL REGISTRY  (version + lineage + approval)   ◄── the junction
              challenger policy: gate ∧ non-inferiority
                              │ approved / @champion flip
                              ▼
              EventBridge (ModelPackage state change) -> deploy Lambda
              -> serverless endpoint config swap (blue/green, NB03 Step 10)
                              │
                              ▼
              Monitoring (NB03 Step 10.2): weekly per-city MAE/MAPE on fresh
              snapshots, drift indices, cold-start & error alarms ──┐
                              ▲                                     │
                              └── monthly retrain cron ◄── drift alarm
                                  (lands a snapshot in trigger/input/ —
                                   scheduled retraining IS this notebook's
                                   trigger, fired by a calendar)
```

Three properties make the loop trustworthy rather than merely automatic. **The registry is the only door to production**: no path — CI/CD, data trigger, or cron — deploys anything except by landing an *approved* version there, so provenance questions always have one place to look, and every challenger (promoted or held) is permanent audit history with its `trigger_key`, `data_version`, and comparison attached. **Retraining is an execution, never an edit**: both triggers start the same parameterized pipeline, so the code that retrains on May data is bit-identical to the code the consistency proof validated. **Every promotion is annotated**: the champion-metric trend on the dashboard carries the `data_version` and version-flip markers, so "the model changed" and "the data changed" are never confounded on the same axis.

**The dashboards, extended for the event-driven layer** (the two panes from Notebook 03 Step 10.2 gain a third):

| Pane | Panels | Alarms wired to it |
|---|---|---|
| **Trigger ops** *(new)* | uploads/day into `trigger/input/`; EventBridge matched-rule count vs Lambda invocations (a gap = permissions/wiring); Lambda errors, duration, DLQ depth; executions by trigger source (loop / Lambda / cron / CI); quarantine rate | DLQ > 0 (page); upload seen but no execution in 10 min; **no-upload heartbeat** > 35 days (the data feed died silently) |
| **Pipeline ops** | execution timeline + step durations; gate pass/fail history; failure reasons by class (data/code/infra) | 2 consecutive failed executions (page); step duration anomaly |
| **Model health** | champion `R²_log`/`MAPE` trend annotated with `data_version` + promotions; per-city MAPE (the Step 4.1 figure, as a time series); challenger-vs-champion decision history | challenger regression streak ≥ 2 (investigate the data source before it wins by attrition); any city > baseline + 5 pts |

The result is the production standard the architecture demands: data arrives, the model refreshes, quality is gated twice (absolute and relative), humans approve exceptions instead of operating the loop, and every screen answers *which model, trained on which bytes, decided this number*.

In [28]:
print("=" * 64)
print("EVENT-DRIVEN RETRAINING — SESSION SUMMARY")
print("=" * 64)
event_df = pd.DataFrame(EVENT_LOG)
cols = [c for c in ["event_id", "file", "trigger", "status", "duration_s",
                    "model_version", "promoted", "failure"] if c in event_df.columns]
print(event_df[cols].to_string(index=False))

print("\nModel Registry after the session:")
for v in sorted(client.search_model_versions(f"name='{REGISTRY_MODEL_NAME}'"),
                key=lambda m: int(m.version)):
    tags = v.tags
    marker = "  <- @champion" if int(v.version) == CHAMPION["version"] else ""
    print(f"  v{v.version}  approval={tags.get('approval_status'):<15} "
          f"data={tags.get('data_version', '-'):<20} "
          f"trigger={tags.get('trigger_key', 'notebook-03'):<28}{marker}")

print(f"\nReigning champion : v{CHAMPION['version']} "
      f"(test_r2_log {CHAMPION['metrics']['test_r2_log']}, "
      f"data {CHAMPION['data_version']})")
print(f"Idempotency ledger: {STATE_FILE} ({len(load_state())} uploads recorded)")
print(f"Quarantine        : {[p.name for p in QUARANTINE_DIR.glob('*.csv')]}")
print(f"Production trigger: lambda_src/handler.py + rule {EVENTBRIDGE_RULE_NAME} (Step 10)")
print()
print("Next: Notebook 04 — Governance & explainability (SHAP), the per-city model card,")
print("      and a permanent policy for the frozen-reference-set evaluation question.")

EVENT-DRIVEN RETRAINING — SESSION SUMMARY
                  event_id                         file       trigger    status  duration_s
evt-20260816-153542-3ffef0 Listings_live_1786722834.csv notebook-loop Succeeded       724.6

Model Registry after the session:


  v1  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v2  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v3  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v4  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v5  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v6  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                 
  v7  approval=approved        data=sha256:097b0bbfea3c  trigger=notebook-03                   <- @champion

Reigning champion : v7 (test_r2_log 0.8708, data sha256:097b0bbfea3c)
Idempotency ledger: processed_uploads.json (3 uploads recorded)
Quarantine        : []
Production trigger: lambda_src/handler.py + rule iti113-team14-airbnb-csv-upload (Step 10)

Next: Notebook 04 — Governance & explainability (SHAP), the per-city model car

---
## Checklist

- [ ] Controlled folder contract in place: watched `trigger/input/`, inert `manual-input/`, `quarantine/`, idempotency ledger
- [ ] `preprocess.py` v1.1 (directory-aware `--input`) written; `pipeline_lib` / `train.py` untouched
- [ ] Manual pipeline test passed **before** any trigger was armed (with a parameter override)
- [ ] Monitoring loop detected the April snapshot and started the pipeline with `InputDataUrl`
- [ ] Gate passed; challenger compared to champion **by written policy**; registered with lineage tags
- [ ] Promotion executed: `@champion` re-pointed; superseded version tagged; MLflow run + figure logged
- [ ] Idempotency demonstrated: re-running the loop re-processed nothing
- [ ] Poison upload failed fast at the contract layer, was quarantined, and its logs diagnosed programmatically
- [ ] AWS forensics cells (pipeline steps, training-job logs, Lambda logs) ready for Studio
- [ ] Production path complete: Lambda handler code, EventBridge rule pattern, least-privilege policy, one-time setup, live test
- [ ] CI/CD + dashboard integration documented: the registry junction, the closed loop, the three dashboard panes